# Assignment 5

## Setup: State, Action Space, and MDP Structure

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import os

# Ensure plots show up inline inside the notebook
%matplotlib inline

# Create outputs directory locally
os.makedirs('./outputs/', exist_ok=True)

STATES  = ['High', 'Low', 'Charging']   # indices 0, 1, 2
ACTIONS = ['Search', 'Wait']            # indices 0, 1
GAMMA   = 0.9
THETA   = 1e-6

# P[s, a, s'] = transition probability
P = np.zeros((3, 2, 3))

# --- High (0) + Search (0) ---
P[0, 0, 0] = 0.7   # High -> High
P[0, 0, 1] = 0.3   # High -> Low

# --- High (0) + Wait (1) ---
P[0, 1, 0] = 1.0   # High -> High

# --- Low (1) + Search (0) ---
P[1, 0, 0] = 0.4   # Low -> High  (recharge penalty)
P[1, 0, 1] = 0.6   # Low -> Low

# --- Low (1) + Wait (1) ---
P[1, 1, 1] = 1.0   # Low -> Low

# --- Charging (2) + Wait (1) ---
P[2, 1, 0] = 1.0   # Charging -> High

# R[s, a] = expected immediate reward
R = np.zeros((3, 2))

R[0, 0] = 4.0
R[0, 1] = 1.0
R[1, 0] = 0.4 * (-3) + 0.6 * 4   # = 1.2
R[1, 1] = 1.0
R[2, 1] = 0.0

print("Transition probability row sums (each must = 1.0):")
for s, sname in enumerate(STATES):
    for a, aname in enumerate(ACTIONS):
        total = P[s, a, :].sum()
        # Charging + Search has no transitions defined -> sum = 0
        if not (s == 2 and a == 0):
            print(f"  P[{sname}, {aname}, :] sum = {total:.1f}")

print("\nReward matrix R[s, a]:")
print(f"{'':12}", end="")
for a in ACTIONS:
    print(f"{a:>10}", end="")
print()
for s, sname in enumerate(STATES):
    print(f"  {sname:<10}", end="")
    for a in range(len(ACTIONS)):
        print(f"{R[s, a]:>10.2f}", end="")
    print()

## Task 1 | Iterative Policy Evaluation

In [ ]:
def policy_evaluation(P, R, policy, gamma, theta):
    n_states = len(STATES)
    V = np.zeros(n_states)
    iteration = 0

    while True:
        delta = 0.0
        V_new = np.zeros(n_states)
        for s in range(n_states):
            a = policy[s]
            # Bellman Expectation
            V_new[s] = np.sum(P[s, a, :] * (R[s, a] + gamma * V))
        delta = np.max(np.abs(V_new - V))
        V = V_new
        iteration += 1
        if delta < theta:
            break

    print(f"  policy_evaluation converged in {iteration} iterations.")
    return V

# Fixed policy: High->Search(0), Low->Wait(1), Charging->Wait(1)
policy_task1 = [0, 1, 1]

print("Policy: High->Search, Low->Wait, Charging->Wait")
V_pi = policy_evaluation(P, R, policy_task1, GAMMA, THETA)
print("\n  Vπ(s):")
for s, sname in enumerate(STATES):
    print(f"    Vπ({sname}) = {V_pi[s]:.6f}")

fig, ax = plt.subplots(figsize=(7, 4))
colors = ['#4C72B0', '#DD8452', '#55A868']
bars = ax.bar(STATES, V_pi, color=colors, edgecolor='black', width=0.5)
ax.bar_label(bars, fmt='%.4f', padding=4)
ax.set_title('Task 1.3 — Vπ(s) under fixed policy', fontsize=12)
ax.set_xlabel('State')
ax.set_ylabel('Value Vπ(s)')
ax.set_ylim(0, max(V_pi) * 1.2)
ax.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig('./outputs/task1_bar.png', dpi=150)
plt.show()

## Task 2 | Value Iteration

In [ ]:
def value_iteration(P, R, gamma, theta):
    n_states  = len(STATES)
    n_actions = len(ACTIONS)
    V = np.zeros(n_states)
    history = [V.copy()]
    iteration = 0

    while True:
        Q = np.zeros((n_states, n_actions))
        for s in range(n_states):
            for a in range(n_actions):
                # 'Search' is NOT allowed in 'Charging' State -> Exclude explicitly via -inf Q-value
                if s == 2 and a == 0:
                    Q[s, a] = -np.inf
                else:    
                    Q[s, a] = np.sum(P[s, a, :] * (R[s, a] + gamma * V))
        V_new = np.max(Q, axis=1)
        delta = np.max(np.abs(V_new - V))
        V = V_new
        history.append(V.copy())
        iteration += 1
        if delta < theta:
            break

    print(f"  value_iteration converged in {iteration} iterations.")
    return V, history

def extract_policy(V_star, P, R, gamma):
    n_states  = len(STATES)
    n_actions = len(ACTIONS)
    policy = np.zeros(n_states, dtype=int)

    for s in range(n_states):
        Q_s = np.zeros(n_actions)
        for a in range(n_actions):
            # Mask invalid actions
            if s == 2 and a == 0:
                Q_s[a] = -np.inf
            else:    
                Q_s[a] = np.sum(P[s, a, :] * (R[s, a] + gamma * V_star))
        policy[s] = np.argmax(Q_s)

    return policy

V_star_vi, vi_history = value_iteration(P, R, GAMMA, THETA)

print("\n  V*(s):")
for s, sname in enumerate(STATES):
    print(f"    V*({sname}) = {V_star_vi[s]:.6f}")

pi_star_vi = extract_policy(V_star_vi, P, R, GAMMA)
print("\n  Optimal policy π*(s):")
for s, sname in enumerate(STATES):
    print(f"    π*({sname}) = {ACTIONS[pi_star_vi[s]]}")

vi_history_arr = np.array(vi_history) 
fig, ax = plt.subplots(figsize=(9, 5))
linestyles = ['-o', '-s', '-^']
for s, sname in enumerate(STATES):
    ax.plot(vi_history_arr[:, s], linestyles[s], label=f'V({sname})', markersize=4)
ax.set_title('Task 2.3 — Value Iteration Convergence')
ax.set_xlabel('Iteration')
ax.set_ylabel('V(s)')
ax.legend()
ax.grid(linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig('./outputs/task2_convergence.png', dpi=150)
plt.show()

## Task 3 | Policy Iteration

In [ ]:
def policy_improvement(V, P, R, gamma, old_policy):
    n_states  = len(STATES)
    n_actions = len(ACTIONS)
    new_policy = np.zeros(n_states, dtype=int)

    for s in range(n_states):
        Q_s = np.zeros(n_actions)
        for a in range(n_actions):
            # Mask invalid actions
            if s == 2 and a == 0:
                Q_s[a] = -np.inf
            else:
                Q_s[a] = np.sum(P[s, a, :] * (R[s, a] + gamma * V))
        new_policy[s] = np.argmax(Q_s)

    stable = np.array_equal(new_policy, old_policy)
    return new_policy, stable

def policy_iteration(P, R, gamma, theta):
    n_states = len(STATES)
    # Start: all states take Wait (index 1)
    policy = np.ones(n_states, dtype=int)
    policy_hist = [policy.copy()]
    V_hist = []

    iteration = 0
    print("  Policy Iteration steps:\n")
    while True:
        iteration += 1
        V = policy_evaluation(P, R, policy, gamma, theta)
        V_hist.append(V.copy())

        print(f"  Iteration {iteration}:")
        print("    Policy:", {STATES[s]: ACTIONS[policy[s]] for s in range(n_states)})
        print("    V:", {STATES[s]: round(V[s], 4) for s in range(n_states)})

        new_policy, stable = policy_improvement(V, P, R, gamma, policy)
        policy_hist.append(new_policy.copy())

        if stable:
            print(f"\n  Policy stable — converged after {iteration} improvement step(s).")
            break
        policy = new_policy

    return policy, V, policy_hist, V_hist

pi_star_pi, V_star_pi, pol_hist, V_hist_pi = policy_iteration(P, R, GAMMA, THETA)

print("\n  Final Optimal Policy π*(s):")
for s, sname in enumerate(STATES):
    print(f"    π*({sname}) = {ACTIONS[pi_star_pi[s]]}")

V_arr = np.array(V_hist_pi)   # shape (n_pi_iters, 3)
fig, ax = plt.subplots(figsize=(8, 5))
for s, sname in enumerate(STATES):
    ax.plot(range(1, len(V_hist_pi) + 1), V_arr[:, s], marker='o', label=f'V({sname})')
ax.set_title('Task 3.3a — Vπ(s) across Policy Iteration steps')
ax.set_xlabel('Policy Iteration Step')
ax.set_ylabel('Value V(s)')
ax.set_xticks(range(1, len(V_hist_pi) + 1))
ax.legend()
ax.grid(linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig('./outputs/task3a_pi_values.png', dpi=150)
plt.show()

policy_matrix = np.array(pol_hist[:-1])   # shape (n_iters, 3)  — action indices
n_iters = policy_matrix.shape[0]

fig, ax = plt.subplots(figsize=(max(5, n_iters * 1.5), 3.5))
cmap = plt.get_cmap('Set2', 2)
im = ax.imshow(policy_matrix.T, aspect='auto', cmap=cmap, vmin=0, vmax=1)

ax.set_yticks(range(len(STATES)))
ax.set_yticklabels(STATES)
ax.set_xticks(range(n_iters))
ax.set_xticklabels([f'Iter {i+1}' for i in range(n_iters)])
ax.set_title('Task 3.3b — Policy at Each Iteration (0 = Search, 1 = Wait)')
ax.set_xlabel('Iteration')
ax.set_ylabel('State')

for i in range(n_iters):
    for s in range(len(STATES)):
        action_name = ACTIONS[policy_matrix[i, s]]
        ax.text(i, s, action_name, ha='center', va='center', fontsize=11, fontweight='bold', color='black')

patch0 = mpatches.Patch(color=cmap(0), label='Search')
patch1 = mpatches.Patch(color=cmap(1), label='Wait')
ax.legend(handles=[patch0, patch1], loc='upper right', bbox_to_anchor=(1.15, 1))
plt.tight_layout()
plt.savefig('./outputs/task3b_policy_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## Bonus Task | Gymnasium Q-Learning
Here we implement the custom MDP environment using `gymnasium` and learn the optimal policy using Tabular Q-learning.

In [ ]:
import gymnasium as gym
from gymnasium import spaces

class RobotBatteryEnv(gym.Env):
    def __init__(self):
        super(RobotBatteryEnv, self).__init__()
        self.observation_space = spaces.Discrete(3)
        self.action_space = spaces.Discrete(2)
        # 0=High, 1=Low, 2=Charging
        self.state = 0
        
    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.state = 0 # Default start state
        return self.state, {}
        
    def step(self, action):
        # Prevent invalid action logic entirely
        if self.state == 2 and action == 0:
            action = 1
            
        prob = np.random.rand()
        
        if self.state == 0: # High
            reward = 4.0 if action == 0 else 1.0
            next_state = 0 if (action == 1 or prob < 0.7) else 1
        elif self.state == 1: # Low
            if action == 0: # Search
                if prob < 0.4:
                    next_state = 0
                    reward = -3.0
                else:
                    next_state = 1
                    reward = 4.0
            else: # Wait
                next_state = 1
                reward = 1.0
        else: # Charging (action is forced to Wait=1)
            next_state = 0
            reward = 0.0
                
        self.state = next_state
        return self.state, reward, False, False, {}

# Instantiate environment
env = RobotBatteryEnv()

# Hyperparameters
alpha = 0.1
gamma = 0.9
epsilon = 0.1
episodes = 50000

# Tabular Q-Table
q_table = np.zeros((3, 2))
q_table[2, 0] = -np.inf # Search in Charging state is disallowed

state, _ = env.reset()

for ep in range(episodes):
    # Epsilon-greedy action selection
    if np.random.rand() < epsilon:
        action = env.action_space.sample()
        if state == 2 and action == 0:
            action = 1 # Force invalid random actions back to Wait
    else:
        action = np.argmax(q_table[state, :])
        
    next_state, reward, _, _, _ = env.step(action)
    
    # Q-Learning Step
    best_next_act = np.argmax(q_table[next_state, :])
    td_target = reward + gamma * q_table[next_state, best_next_act]
    td_error = td_target - q_table[state, action]
    q_table[state, action] += alpha * td_error
    
    state = next_state

print("== Tabular Q-Learning Final Policy ==")
learned_policy = np.argmax(q_table, axis=1)
for s, sname in enumerate(STATES):
    print(f"  π_Q({sname}) = {ACTIONS[learned_policy[s]]}   | Q-values: Search={q_table[s, 0]:.4f}, Wait={q_table[s, 1]:.4f}")

# Compare to Analytical Policy from Value/Policy Iteration
print("\nComparison to Analytical Policy (Value Iteration):")
match = np.array_equal(learned_policy, pi_star_vi)
print(f"Does the Q-Learning policy match the Analytical Optimal Policy? {'Yes ✓' if match else 'No ✗'}")


## Task 4 | Analysis and Interpretation

### 4.1 Comparing Value Iteration vs Policy Iteration
- **Value Iteration** typically takes many sweeps as each sweep improves the value function incrementally. It directly uses the Bellman Optimality operator to find the optimal values without explicitly evaluating a policy at step.
- **Policy Iteration** converges in very few outer steps (often 2–4). However, each step contains a full Policy Evaluation that takes many internal sweeps. Policy Iteration acts greedily on correctly evaluated policies making it stabilize faster per outer iteration on small state spaces.

### 4.2 Convergence Behaviour
- Under **Value Iteration**, `V(High)` shoots up quickly given the high expected search reward (+4). `V(Charging)` scales up linearly as the model correlates it to the valuable `High` escape path.
- **Value Iteration** evaluates the Bellman *Optimality* operator (takes the true max of future actions at each sweep) causing distinct curve convergence compared to *Policy Evaluation* which relies on expectation over a fixed rule constraint point.

### 4.3 Optimal Policy Interpretation
- **High `->` Search**: When the battery is full, `Search` expected value (+4) vastly exceeds `Wait` (+1) with minor risk of draining to `Low`.
- **Low `->` Search**: The 40% chance of sudden battery depletion (Recharge Penalty) does not statistically outweigh the 60% chance of successful extraction (+4) in expected value calculations. The Q-value calculation proves it mathematically beats pure waiting.
- **Charging `->` Wait**: Only `Wait` is permitted (since `Search` is explicitly disallowed and masked by `-np.inf` Q-value), returning the robot to `High` safely.

### 4.4 Practical Insight
- Devices must balance greedy resource utilization and preservation conservatively. By penalizing depleted battery states explicitly but rewarding standard operations, RL MDP policies mimic biological 'forage-delay-recharge' cycles. Firmwares can implement this directly as a discrete, zero-compute `[Search, Search, Wait]` dictionary map lookup!